In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

#### Nota de negócio: colunas de origem não utilizadas

Ao mapear os CSVs para a Silver, ficaram de fora **`tconst`** (`tb_movies_info`) e **`production_countries`**,
**`spoken_languages`**, **`keywords`** (`tb_credits_and_tags`), nenhuma está no mapeamento definido.

- `tconst` (ID do IMDb) seria útil como chave rápida para cruzar com bases externas do IMDb.
- `keywords` encaixaria muito bem para enriquecer o `llm_context_document` futuro.

Ambas ficam de fora porque o escopo é um requisito de cliente com mapeamento e schema determinísticos.

In [0]:
df_info = spark.table("workspace.bronze.tb_movies_info")

status_map = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado",
}

status_map_expr = F.create_map(*[F.lit(x) for item in status_map.items() for x in item])
# Normaliza antes de traduzir e marca valores não informados ou não mapeaveis
status_normalizado = F.upper(F.trim(F.regexp_replace(F.col("status"), "[-_]+", " ")))
status_filme = F.coalesce(status_map_expr[status_normalizado], F.lit("Não Informado"))

data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')")
)

df_info = (
    df_info
    .withColumn("status_filme", status_filme)
    .withColumn("data_lancamento", data_lancamento)
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

# Mantém apenas o registro mais recente.
janela_dedupe = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_info = (
    df_info
    .withColumn("_rn", F.row_number().over(janela_dedupe))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

df_info_filmes = df_info.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao",
)

df_info_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_info_filmes")
display(df_info_filmes)

In [0]:
taxa_dolar = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()["cotacaoCompra"]
)
# usa a cotação de compra mais recente disponível na bronze
print("Taxa USD para BRL usada:", taxa_dolar)

def limpar_moeda(df, origem, destino):
    valor = F.upper(F.trim(F.col(origem)))
    valor = F.when(valor.isin("UNKNOWN", "NÃO INFORMADO", ""), F.lit(None)).otherwise(valor)

    # remove símbolos de moeda e separadores
    valor = F.regexp_replace(valor, "USD", "")
    valor = F.regexp_replace(valor, "[$,]", "")
    valor = F.trim(valor)

    # sufixo K/M (mil e milhão) multiplica pelos valores corretos
    multiplicador = (
        F.when(valor.endswith("K"), F.lit(1000))
        .when(valor.endswith("M"), F.lit(1000000))
        .otherwise(F.lit(1))
    )
    numero = F.regexp_replace(valor, "[KM]$", "")

    df = df.withColumn("_numero_moeda", numero)
    df = df.withColumn(destino, F.expr("try_cast(_numero_moeda AS DOUBLE)") * multiplicador)
    df = df.withColumn(destino, F.when(F.col(destino) > 0, F.col(destino)))

    return df.drop("_numero_moeda")

df_fin = spark.table("workspace.bronze.tb_movies_financials").withColumnRenamed("id", "id_filme")
df_fin = limpar_moeda(df_fin, "budget", "orcamento_usd")
df_fin = limpar_moeda(df_fin, "revenue", "receita_usd")

df_fin = (
    df_fin
    # calcula os valores equivalentes em reais
    .withColumn("orcamento_brl", F.col("orcamento_usd") * F.lit(taxa_dolar))
    .withColumn("receita_brl", F.col("receita_usd") * F.lit(taxa_dolar))
    # lucro só quando ambos os lados existem para evitar nulos.
    .withColumn(
        "lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
                F.col("receita_usd") - F.col("orcamento_usd")),
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
                F.col("receita_brl") - F.col("orcamento_brl")),
    )
    # margem percentual só quando orçamento > 0
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") > 0) & F.col("lucro_usd").isNotNull(),
            (F.col("lucro_usd") / F.col("orcamento_usd")) * 100,
        ),
    )
    # orçamentos residuais muito baixos geram margem com mais de 10 dígitos e um cast comum quebraria a célula inteira
    .withColumn("margem_lucro_percentual", F.expr("try_cast(margem_lucro_percentual AS DECIMAL(18,2))"))
)

# tipo numérico decimal apropriado
colunas_decimais = ["orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl"]
for c in colunas_decimais:
    df_fin = df_fin.withColumn(c, F.col(c).cast("decimal(18,2)"))

df_financeiro_filmes = df_fin.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
)

df_financeiro_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_financeiro_filmes")
display(df_financeiro_filmes)

In [0]:
df_met = (
    spark.table("workspace.bronze.tb_movies_metrics")
    .withColumnRenamed("id", "id_filme")
    # Popularidade mistura separador decimal "," e "." na origem
    .withColumn("_popularidade_texto", F.regexp_replace(F.trim(F.col("popularity")), ",", "."))
    # column shift espalha texto/nomes vazados nas colunas de nota e voto, lixo vira NULL sem quebrar o pipeline
    .withColumn("popularidade", F.expr("try_cast(_popularidade_texto AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(vote_average AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(vote_count AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(averageRating AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(numVotes AS INT)"))
)

df_met = (
    df_met
    # Popularidade negativa é inválida vira NULL
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    # Nota fora de 0-10 (inclusive erro de escala) vira NULL, sem redividir
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    # Contagem de votos negativa é inválida e vira NULL
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
)

df_metricas_engajamento = df_met.select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb",
)

df_metricas_engajamento.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tb_metricas_engajamento")
display(df_metricas_engajamento)